In [1]:
# ===============================================================
# Cell 1: 環境設定與套件安裝
# ===============================================================
import sys
import subprocess
import importlib
import logging

# 設定日誌
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', handlers=[
    logging.StreamHandler(sys.stdout)
])

def setup_environment():
    """
    檢查並安裝所有必要的 Python 套件。
    """
    # 套件列表 (名稱, 導入時檢查的名稱)
    packages = [
        ('requests', 'requests'),
        ('beautifulsoup4', 'bs4'),
        ('pandas', 'pandas'),
        ('lxml', 'lxml')
    ]
    logging.info("--- 正在檢查並安裝必要套件 ---")
    
    all_installed = True
    for pkg_name, import_name in packages:
        try:
            importlib.import_module(import_name)
            logging.info(f"✅ {pkg_name} (導入名: {import_name}) 已正確導入。")
        except ImportError:
            all_installed = False
            logging.warning(f"--- 正在安裝 {pkg_name} ---")
            try:
                # 使用 pip 來安裝套件
                subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])
                logging.info(f"✅ {pkg_name} 安裝完成！")
            except subprocess.CalledProcessError as e:
                logging.error(f"🔥 安裝 {pkg_name} 失敗: {e}")
                logging.error("請手動在您的終端機執行: pip install requests beautifulsoup4 pandas lxml")
                sys.exit(1) # 安裝失敗，停止執行

    if all_installed:
        logging.info("\n🎉 所有必要的套件都已準備就緒！")
    else:
        logging.info("\n🎉 套件安裝完成！")

# 立即執行環境設定
setup_environment()

2025-10-27 16:43:04,551 - INFO - --- 正在檢查並安裝必要套件 ---
2025-10-27 16:43:04,609 - INFO - ✅ requests (導入名: requests) 已正確導入。
2025-10-27 16:43:04,645 - INFO - ✅ beautifulsoup4 (導入名: bs4) 已正確導入。
2025-10-27 16:43:04,645 - INFO - ✅ pandas (導入名: pandas) 已正確導入。
2025-10-27 16:43:04,645 - INFO - ✅ lxml (導入名: lxml) 已正確導入。
2025-10-27 16:43:04,645 - INFO - 
🎉 所有必要的套件都已準備就緒！


In [2]:
# ===============================================================
# Cell 2: 匯入套件、全域變數和輔助函式
# ===============================================================
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timezone, timedelta
import os
import sys
import json
import re
import time
import lxml  # lxml 是 BeautifulSoup 會使用的高效能剖析器
import logging # 繼續使用日誌

# --- 全域變數 ---
# *** 這非常重要：檔案名稱已修改為您上傳的檔案名稱 ***
DB_FILENAME = 'news_snapshot_log.csv'           # 主要資料庫檔案名稱 (您的舊檔案)
TEMP_JSON_FILENAME = 'temp_all_headlines.json'  # 暫存的頭條JSON檔案

# 假裝是瀏覽器，避免被 403 (禁止存取) 錯誤
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}

# *** 這非常重要：欄位已完全比對您上傳的 news_snapshot_log.csv 檔案 ***
COLUMNS = [
    'url', 
    'title', 
    'content', 
    'source',          # 來源代碼: CNA, UDN, SETN, TVBS, EBC, NOWNEWS, ETTODAY, LTN, CHINATIMES
    'published_at', 
    'scraped_at', 
    'credibility_label', # 信用評等 (0 或 1)
    'headline_level'   # 頭條等級 (1, 2, 3)
] # 總共 8 個欄位，移除了 'source_name' 以匹配您的檔案

# --- 輔助函式 ---

def get_soup(url):
    """輔助函式：發送請求並取得 BeautifulSoup 物件"""
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status() # 如果請求失敗 (例如 404, 500) 會拋出錯誤
        return BeautifulSoup(response.text, 'lxml')
    except requests.RequestException as e:
        logging.warning(f"🟡 抓取 {url} 失敗: {e}")
        return None

def normalize_url(base_url, link):
    """輔助函式：將相對路徑 URL (例如 /news/abc) 轉換為絕對路徑"""
    if not link:
        return None
    link = link.strip() # 移除多餘空白
    if link.startswith('//'):
        return 'https:' + link
    if link.startswith('/'):
        return base_url.rstrip('/') + link
    if link.startswith('http'):
        return link
    if not link.startswith('http'):
        return base_url.rstrip('/') + '/' + link.lstrip('/')
    return link

def safe_find_text(soup, selector, default='N/A'):
    """輔助函式：安全地尋找元素並取得文字，若找不到則回傳預設值"""
    if not soup:
        return default
    element = soup.select_one(selector)
    if element:
        return element.get_text(separator='\n', strip=True)
    return default

def get_current_utc_time():
    """取得目前 UTC+8 (台灣時間) 的時間字串"""
    taiwan_tz = timezone(timedelta(hours=8))
    return datetime.now(taiwan_tz).strftime("%Y-%m-%d %H:%M:%S.%f")

def remove_duplicates(headlines):
    """移除重複的頭條，保留 URL 唯一，並優先取等級 (level) 最高的"""
    if not headlines: 
        return []
    
    df = pd.DataFrame(headlines)
    df = df.sort_values('headline_level', ascending=False)
    # 移除 URL + Title 都重複的 (避免同 URL 但標題被稍改)
    df_unique = df.drop_duplicates(subset=['url', 'title'], keep='first')
    # 再次移除 URL 重複，確保 URL 絕對唯一
    df_unique = df_unique.drop_duplicates(subset=['url'], keep='first')
    
    return df_unique.to_dict('records')

logging.info("✅ Cell 2 輔助函式已載入")

2025-10-27 16:43:08,591 - INFO - ✅ Cell 2 輔助函式已載入


In [3]:
# ===============================================================
# Cell 3: Part 1 - 抓取所有網站的頭條新聞 (函式定義)
# ===============================================================

def fetch_cna_headlines():
    """(1/9) 抓取中央社 (CNA) 頭條"""
    logging.info("--- (1/9) 正在抓取 CNA 中央社頭條 ---")
    base_url = 'https://www.cna.com.tw'
    soup = get_soup(base_url + '/list/aall.aspx') 
    if not soup:
        return []

    headlines = []
    scraped_at = get_current_utc_time()

    # Level 3: 頁面頂部的「重點新聞」
    for item in soup.select('div.main-topic ul li a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'www.cna.com.tw' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'CNA'})
        except Exception as e:
            logging.warning(f"🟡 CNA Level 3 剖析錯誤: {e}")

    # Level 1: 即時新聞列表
    for item in soup.select('ul.main-list li a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'www.cna.com.tw' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'CNA'})
        except Exception as e:
            logging.warning(f"🟡 CNA Level 1 剖析錯誤: {e}")
            
    logging.info(f"💡 CNA 抓取到 {len(headlines)} 筆頭條")
    return headlines

def fetch_udn_headlines():
    """(2/9) 抓取聯合新聞網 (UDN) 頭條"""
    logging.info("--- (2/9) 正在抓取 UDN 聯合新聞網頭條 ---")
    base_url = 'https://udn.com'
    soup = get_soup(base_url + '/news/breaknews/1') # 抓取即時新聞總覽
    if not soup:
        return []

    headlines = []
    scraped_at = get_current_utc_time()

    # Level 3: UDN 即時新聞頁的「要聞」
    for item in soup.select('div.context-box__content--main h2 a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'udn.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'UDN'})
        except Exception as e:
            logging.warning(f"🟡 UDN Level 3 剖析錯誤: {e}")

    # Level 1: UDN 即時新聞列表 (Level 1)
    for item in soup.select('div.story-list__news div.story-list__info h2 a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'udn.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'UDN'})
        except Exception as e:
            logging.warning(f"🟡 UDN Level 1 剖析錯誤: {e}")

    logging.info(f"💡 UDN 抓取到 {len(headlines)} 筆頭條")
    return headlines

def fetch_setn_headlines():
    """(3/9) 抓取三立新聞網 (SETN) 頭條"""
    logging.info("--- (3/9) 正在抓取 SETN 三立新聞網頭條 ---")
    base_url = 'https://www.setn.com'
    soup = get_soup(base_url)
    if not soup:
        return []

    headlines = []
    scraped_at = get_current_utc_time()

    # Level 3: 焦點新聞輪播
    for item in soup.select('div.focus_news div.captionText a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'setn.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'SETN'})
        except Exception as e:
            logging.warning(f"🟡 SETN Level 3 剖析錯誤: {e}")

    # Level 2: 熱門新聞
    for item in soup.select('div.top-hot-list li a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'setn.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 2, 'scraped_at': scraped_at, 'source': 'SETN'})
        except Exception as e:
            logging.warning(f"🟡 SETN Level 2 剖析錯誤: {e}")

    # Level 1: 即時新聞列表
    for item in soup.select('div.immediate-news-area div.news-list li a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'setn.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'SETN'})
        except Exception as e:
            logging.warning(f"🟡 SETN Level 1 剖析錯誤: {e}")
            
    logging.info(f"💡 SETN 抓取到 {len(headlines)} 筆頭條")
    return headlines

def fetch_tvbs_headlines():
    """(4/9) 抓取 TVBS 新聞網頭條"""
    logging.info("--- (4/9) 正在抓取 TVBS 新聞網頭條 ---")
    base_domain = 'https://www.tvbs.com.tw' # 首頁的 URL
    soup = get_soup(base_domain) # 抓取 www.tvbs.com.tw 首頁
    if not soup:
        return []

    headlines = []
    scraped_at = get_current_utc_time()

    # Level 3: 英雄區主輪播
    for item in soup.select('div.hero_main a.swiper-slide'):
        try:
            title = item.select_one('img')['alt']
            url = normalize_url(base_domain, item.get('href')) 
            if title and url:
                headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'TVBS'})
        except Exception as e:
            logging.warning(f"🟡 TVBS Level 3 剖析錯誤: {e}")

    # Level 2: 英雄區副新聞
    for item in soup.select('div.hero_sub a.sub_item'):
        try:
            title = item.select_one('h3').get_text(strip=True)
            url = normalize_url(base_domain, item.get('href'))
            if title and url and 'news.tvbs.com.tw' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 2, 'scraped_at': scraped_at, 'source': 'TVBS'})
        except Exception as e:
            logging.warning(f"🟡 TVBS Level 2 剖析錯誤: {e}")

    # Level 1: TVBS 新聞區塊 (Breaking News)
    for item in soup.select('div.section_news a.news_popular'):
        try:
            title = item.select_one('h2').get_text(strip=True)
            url = normalize_url(base_domain, item.get('href'))
            if title and url and 'news.tvbs.com.tw' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'TVBS'})
        except Exception as e:
            logging.warning(f"🟡 TVBS Level 1 剖析錯誤: {e}")
            
    logging.info(f"💡 TVBS 抓取到 {len(headlines)} 筆頭條")
    return headlines

def fetch_ebc_headlines():
    """(5/9) 抓取東森新聞 (EBC) 頭條"""
    logging.info("--- (5/9) 正在抓取 EBC 東森新聞頭條 ---")
    base_url = 'https://news.ebc.net.tw'
    soup = get_soup(base_url)
    if not soup:
        return []

    headlines = []
    scraped_at = get_current_utc_time()

    # Level 3: 頂部輪播 (latest_news)
    for item in soup.select('div.latest_news a.swiper-slide'):
        try:
            title_element = item.select_one('h3.slide_title')
            title = item.get('data-title') or (title_element.get_text(strip=True) if title_element else None)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'ebc.net.tw' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'EBC'})
        except Exception as e:
            logging.warning(f"🟡 EBC Level 3 剖析錯誤: {e}")

    # Level 2: 熱門新聞 (hot_news)
    for item in soup.select('div.hot_news div.list_slider li a'):
        try:
            title = item.select_one('h3.item_title').get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'ebc.net.tw' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 2, 'scraped_at': scraped_at, 'source': 'EBC'})
        except Exception as e:
            logging.warning(f"🟡 EBC Level 2 剖析錯誤: {e}")

    # Level 1: 焦點推薦 (focus_box)
    for item in soup.select('div.focus_box div.section_content a.item'):
        try:
            title = item.select_one('h3.item_title').get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'ebc.net.tw' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'EBC'})
        except Exception as e:
            logging.warning(f"🟡 EBC Level 1 剖析錯誤: {e}")
            
    logging.info(f"💡 EBC 抓取到 {len(headlines)} 筆頭條")
    return headlines

def fetch_nownews_headlines():
    """(6/9) 抓取 NOWnews 今日新聞頭條"""
    logging.info("--- (6/9) 正在抓取 NOWnews 今日新聞頭條 ---")
    base_url = 'https://www.nownews.com'
    soup = get_soup(base_url)
    if not soup:
        return []

    headlines = []
    scraped_at = get_current_utc_time()

    # Level 3: 焦點新聞輪播
    for item in soup.select('div#focusSlider div.slider-item a'):
        try:
            title = item.select_one('figcaption').get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'nownews.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'NOWNEWS'})
        except Exception as e:
            logging.warning(f"🟡 NOWnews Level 3 剖析錯誤: {e}")

    # Level 2: 右側熱門新聞
    for item in soup.select('ul.hotnews-wrap li a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'nownews.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 2, 'scraped_at': scraped_at, 'source': 'NOWNEWS'})
        except Exception as e:
            logging.warning(f"🟡 NOWnews Level 2 剖析錯誤: {e}")

    # Level 1: 焦點區塊 (在熱門下方)
    for item in soup.select('div.nnBlk.focus li.item a'):
        try:
            title = item.select_one('h3.title').get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'nownews.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'NOWNEWS'})
        except Exception as e:
            logging.warning(f"🟡 NOWnews Level 1 剖析錯誤: {e}")
            
    logging.info(f"💡 NOWnews 抓取到 {len(headlines)} 筆頭條")
    return headlines

def fetch_chinatimes_headlines():
    """(7/9) 抓取中時新聞網 (China Times) 頭條"""
    logging.info("--- (7/9) 正在抓取 China Times 中時新聞網頭條 ---")
    base_url = 'https://www.chinatimes.com'
    soup = get_soup(base_url)
    if not soup:
        return []

    headlines = []
    scraped_at = get_current_utc_time()

    # Level 3: 焦點大圖 (focus-gallery)
    for item in soup.select('div.focus-gallery ul.item-group li.item-entry a'):
        try:
            title_element = item.select_one('h3.caption-title')
            title = title_element.get_text(separator=' ', strip=True) if title_element else None
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'chinatimes.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'CHINATIMES'})
        except Exception as e:
            logging.warning(f"🟡 China Times Level 3 剖析錯誤: {e}")

    # Level 2: 熱門新聞 (hot-news)
    for item in soup.select('section.hot-news ul.vertical-list li h4.title a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'chinatimes.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 2, 'scraped_at': scraped_at, 'source': 'CHINATIMES'})
        except Exception as e:
            logging.warning(f"🟡 China Times Level 2 剖析錯誤: {e}")

    # Level 1: 即時新聞 (latest-news)
    for item in soup.select('section.latest-news ul.vertical-list li h4.title a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'chinatimes.com' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'CHINATIMES'})
        except Exception as e:
            logging.warning(f"🟡 China Times Level 1 剖析錯誤: {e}")
            
    logging.info(f"💡 China Times 抓取到 {len(headlines)} 筆頭條")
    return headlines

def fetch_ettoday_headlines():
    """(8/9) 抓取 ETtoday 新聞雲頭條 (來自您的 .ipynb)"""
    logging.info("--- (8/9) 正在抓取 ETtoday 新聞雲頭條 ---")
    base_url = 'https://www.ettoday.net'
    soup = get_soup(base_url + '/news/list.htm') # 抓取即時新聞列表
    if not soup:
        return []

    headlines = []
    scraped_at = get_current_utc_time()

    # Level 3: 焦點新聞 (c1)
    for item in soup.select('div.c1 > h3 > a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'ettoday.net' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'ETTODAY'})
        except Exception as e:
            logging.warning(f"🟡 ETtoday Level 3 剖析錯誤: {e}")

    # Level 1: 即時新聞列表 (part_list_2)
    for item in soup.select('div.part_list_2 > div > h3 > a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'ettoday.net' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'ETTODAY'})
        except Exception as e:
            logging.warning(f"🟡 ETtoday Level 1 剖析錯誤: {e}")
            
    logging.info(f"💡 ETtoday 抓取到 {len(headlines)} 筆頭條")
    return headlines

def fetch_ltn_headlines():
    """(9/9) 抓取自由時報 (LTN) 頭條 (來自您的 .ipynb)"""
    logging.info("--- (9/9) 正在抓取 LTN 自由時報頭條 ---")
    base_url = 'https://www.ltn.com.tw'
    soup = get_soup('https://news.ltn.com.tw/list/breakingnews') # 改抓即時新聞頁
    if not soup:
        return []

    headlines = []
    scraped_at = get_current_utc_time()

    # Level 3: 焦點新聞 (focus)
    for item in soup.select('div.focus-news ul.list li a'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'ltn.com.tw' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 3, 'scraped_at': scraped_at, 'source': 'LTN'})
        except Exception as e:
            logging.warning(f"🟡 LTN Level 3 剖析錯誤: {e}")

    # Level 1: 即時新聞列表 (list)
    for item in soup.select('ul.list li a.title'):
        try:
            title = item.get_text(strip=True)
            url = normalize_url(base_url, item.get('href'))
            if title and url and 'ltn.com.tw' in url:
                headlines.append({'url': url, 'title': title, 'headline_level': 1, 'scraped_at': scraped_at, 'source': 'LTN'})
        except Exception as e:
            logging.warning(f"🟡 LTN Level 1 剖析錯誤: {e}")
            
    logging.info(f"💡 LTN 抓取到 {len(headlines)} 筆頭條")
    return headlines

logging.info("✅ Cell 3 (共 9 個) 爬蟲函式已載入")

2025-10-27 16:43:22,729 - INFO - ✅ Cell 3 (共 9 個) 爬蟲函式已載入


In [4]:
# ===============================================================
# Cell 4: Part 1 - 主執行函式 (抓取頭條並儲存)
# ===============================================================

def run_part1_headline_fetch():
    """執行所有頭條抓取，並儲存到暫存 JSON 檔案"""
    logging.info("--- Part 1 開始：抓取所有 9 個網站頭條 ---")
    
    # 執行所有爬蟲
    all_headlines = []
    all_headlines.extend(fetch_cna_headlines())
    all_headlines.extend(fetch_udn_headlines())
    all_headlines.extend(fetch_setn_headlines())
    all_headlines.extend(fetch_tvbs_headlines())
    all_headlines.extend(fetch_ebc_headlines())
    all_headlines.extend(fetch_nownews_headlines())
    all_headlines.extend(fetch_chinatimes_headlines())
    all_headlines.extend(fetch_ettoday_headlines())
    all_headlines.extend(fetch_ltn_headlines())
    
    logging.info(f"\n--- 總共抓取到 {len(all_headlines)} 筆頭條 (含重複) ---")
    
    # 移除重複，保留等級最高的
    unique_headlines = remove_duplicates(all_headlines)
    
    logging.info(f"--- 移除重複後，剩餘 {len(unique_headlines)} 筆獨立頭條 ---")

    # 儲存到 JSON 檔案 (用於 Part 2)
    try:
        with open(TEMP_JSON_FILENAME, 'w', encoding='utf-8') as f:
            json.dump(unique_headlines, f, ensure_ascii=False, indent=4)
        logging.info(f"✅ Part 1 完成：已將頭條暫存於 {TEMP_JSON_FILENAME}")
    except IOError as e:
        logging.error(f"🔥 Part 1 錯誤：無法寫入 JSON 檔案: {e}")
    except Exception as e:
        logging.error(f"🔥 Part 1 發生未知錯誤: {e}")

logging.info("✅ Cell 4 執行函式已載入")

2025-10-27 16:43:42,277 - INFO - ✅ Cell 4 執行函式已載入


In [5]:
# ===============================================================
# Cell 5: Part 2 - 抓取各網站「文章內文」 (函式定義)
# ===============================================================

def scrape_article_content_base(url, content_selector, time_selector):
    """
    抓取文章內文的通用基礎函式 (僅抓取內容和時間)
    """
    soup = get_soup(url)
    if not soup:
        return 'N/A', get_current_utc_time()
    
    # 1. 抓取內容
    content = safe_find_text(soup, content_selector, default='N/A')
    
    # 2. 抓取時間
    published_at = safe_find_text(soup, time_selector, default=get_current_utc_time()) 
    
    # 嘗試清理時間字串 (移除 "發布時間：" 等)
    published_at = re.sub(r'^(發布時間|更新時間)：', '', published_at).strip()
    
    # 嘗試抓取標準日期時間格式 YYYY-MM-DD HH:MM 或 YYYY/MM/DD HH:MM
    match = re.search(r'(\d{4}[-/]\d{2}[-/]\d{2}\s+\d{2}:\d{2})', published_at)
    if match:
        published_at = match.group(1).replace('/', '-') # 統一格式
    else:
        # 如果找不到，就用當前抓取時間作為備案
        published_at = get_current_utc_time()
            
    return content, published_at

def scrape_cna_article(url):
    """CNA 文章頁面爬蟲"""
    content, time = scrape_article_content_base(
        url,
        content_selector='div.article-body',
        time_selector='div.update-time'
    )
    return content, time, 1 # 信用評等 1

def scrape_udn_article(url):
    """UDN 文章頁面爬蟲"""
    content, time = scrape_article_content_base(
        url,
        content_selector='section.article-content__editor',
        time_selector='div.article-content__time'
    )
    return content, time, 1 # 信用評等 1

def scrape_setn_article(url):
    """SETN 文章頁面爬蟲"""
    content, time = scrape_article_content_base(
        url,
        content_selector='div#Content1',
        time_selector='time.page-date'
    )
    return content, time, 0 # 信用評等 0

def scrape_tvbs_article(url):
    """TVBS 文章頁面爬蟲"""
    content, time = scrape_article_content_base(
        url,
        content_selector='div.article_content',
        time_selector='div.time'
    )
    return content, time, 0 # 信用評等 0

def scrape_ebc_article(url):
    """EBC 文章頁面爬蟲"""
    content, time = scrape_article_content_base(
        url,
        content_selector='div.article-main-content',
        time_selector='span.date'
    )
    return content, time, 0 # 信用評等 0

def scrape_nownews_article(url):
    """NOWnews 文章頁面爬蟲"""
    soup = get_soup(url)
    if not soup:
        return 'N/A', get_current_utc_time(), 0

    content_element = soup.select_one('div.article-content')
    content = 'N/A'
    if content_element:
        # 移除所有 'figure' (圖片) 和 'div.media-tool' (分享按鈕)
        for figure in content_element.select('figure'):
            figure.decompose()
        for tool in content_element.select('div.media-tool'):
            tool.decompose()
        content = content_element.get_text(separator='\n', strip=True)

    time = safe_find_text(soup, 'time.date', default=get_current_utc_time())
    match = re.search(r'(\d{4}[-/]\d{2}[-/]\d{2}\s+\d{2}:\d{2})', time)
    if match:
        time = match.group(1).replace('/', '-')
    else:
        time = get_current_utc_time()
        
    return content, time, 0 # 信用評等 0

def scrape_chinatimes_article(url):
    """China Times 文章頁面爬蟲"""
    content, time = scrape_article_content_base(
        url,
        content_selector='div.article-body',
        time_selector='div.meta-info time'
    )
    return content, time, 1 # 信用評等 1

def scrape_ettoday_article(url):
    """ETtoday 文章頁面爬蟲 (來自您的 .ipynb 邏輯)"""
    content, time = scrape_article_content_base(
        url,
        content_selector='div.story',
        time_selector='time.date'
    )
    
    # ETtoday 內容可能包含垃圾訊息，嘗試清理
    if content != 'N/A':
        content_lines = content.split('\n')
        cleaned_lines = [line for line in content_lines if '▲' not in line and '▼' not in line and 'ETtoday' not in line]
        content = '\n'.join(cleaned_lines).strip()

    return content, time, 0 # 信用評等 0

def scrape_ltn_article(url):
    """LTN 文章頁面爬蟲 (來自您的 .ipynb 邏輯)"""
    content, time = scrape_article_content_base(
        url,
        content_selector='div[itemprop="articleBody"]',
        time_selector='span.time'
    )
    
    # 自由時報內容可能包含垃圾訊息，嘗試清理
    if content != 'N/A':
        content_lines = content.split('\n')
        # 移除廣告和導覽連結
        cleaned_lines = [line for line in content_lines if '相關新聞' not in line and '不用抽' not in line and '點我下載' not in line and '想看更多' not in line]
        content = '\n'.join(cleaned_lines).strip()
        
    return content, time, 0 # 您的版本是 0，這裡保持一致

def scrape_article(url, source):
    """爬蟲路由器：根據來源 (source) 呼叫對應的爬蟲函式"""
    try:
        if source == 'CNA':
            return scrape_cna_article(url)
        elif source == 'UDN':
            return scrape_udn_article(url)
        elif source == 'SETN':
            return scrape_setn_article(url)
        elif source == 'TVBS':
            if 'news.tvbs.com.tw' in url:
                return scrape_tvbs_article(url)
            else:
                logging.warning(f"🟡 TVBS 網址非新聞站 ({url})，跳過。")
                return 'N/A', get_current_utc_time(), 0
        elif source == 'EBC':
            return scrape_ebc_article(url)
        elif source == 'NOWNEWS':
            return scrape_nownews_article(url)
        elif source == 'CHINATIMES':
            return scrape_chinatimes_article(url)
        elif source == 'ETTODAY':
            return scrape_ettoday_article(url)
        elif source == 'LTN':
            return scrape_ltn_article(url)
        else:
            logging.warning(f"🟡 未知的來源: {source}，無法抓取 {url}")
            return 'N/A', get_current_utc_time(), 0
    except Exception as e:
        logging.error(f"🔥 處理 {url} 時發生嚴重錯誤: {e}")
        return 'N/A', get_current_utc_time(), 0

logging.info("✅ Cell 5 (共 9 個) 文章內文爬蟲函式已載入")

2025-10-27 16:43:53,341 - INFO - ✅ Cell 5 (共 9 個) 文章內文爬蟲函式已載入


In [6]:
# ===============================================================
# Cell 6: Part 2 - 主執行函式 (抓取新文章並附加到 DB)
# ===============================================================

def run_part2_scrape_and_append():
    """
    讀取 JSON，抓取所有頭條的內文，並將這批新的快照
    **附加(Append)** 到現有的 CSV 資料庫 (news_snapshot_log.csv)。
    """
    logging.info(f"\n--- Part 2 開始：抓取文章內文並附加到 {DB_FILENAME} ---")
    
    # 1. 檢查主資料庫是否存在，這將決定我們是否需要寫入 Header
    file_exists = os.path.exists(DB_FILENAME)
    if file_exists:
        logging.info(f"💡 偵測到現有資料庫 {DB_FILENAME}。新資料將會附加在檔案結尾。")
    else:
        logging.info(f"💡 找不到資料庫 {DB_FILENAME}。將會建立新檔案並寫入欄位名稱。")

    # 2. 讀取 Part 1 抓到的頭條 JSON
    try:
        with open(TEMP_JSON_FILENAME, 'r', encoding='utf-8') as f:
            headlines_to_scrape = json.load(f)
        logging.info(f"💡 已從 {TEMP_JSON_FILENAME} 載入 {len(headlines_to_scrape)} 筆頭條準備抓取。")
    except FileNotFoundError:
        logging.error(f"🔥 Part 2 錯誤：找不到 {TEMP_JSON_FILENAME}。請先執行 Part 1。")
        return
    except json.JSONDecodeError:
        logging.error(f"🔥 Part 2 錯誤：{TEMP_JSON_FILENAME} 檔案格式錯誤。")
        return

    if not headlines_to_scrape:
        logging.warning("👍 Part 2 完成：沒有從 Part 1 接收到任何頭條，無新資料寫入。")
        return

    logging.info(f"--- 開始抓取 {len(headlines_to_scrape)} 筆文章內文 ---")
    
    # 3. 抓取所有文章 (這就是一個完整的「快照」)
    new_snapshot_data = []
    count = 0
    for headline in headlines_to_scrape:
        count += 1
        url = headline['url']
        source = headline['source']
        logging.info(f"  ({count}/{len(headlines_to_scrape)}) 正在抓取 [{source}] {url}...")
        
        try:
            content, pub_time, cred_label = scrape_article(url, source)
            
            if content != 'N/A' and content.strip(): # 確保內容不是 N/A 也不是空字串
                new_snapshot_data.append({
                    'url': url,
                    'title': headline['title'],
                    'content': content,
                    'source': source,
                    'published_at': pub_time,
                    'scraped_at': headline['scraped_at'],
                    'credibility_label': cred_label,
                    'headline_level': headline['headline_level']
                })
                # 增加延遲，避免對伺服器造成太大壓力
                time.sleep(0.2) # 休息 0.2 秒
            else:
                logging.warning(f"  🟡 抓取失敗或內容為空，跳過: {url}")
        except Exception as e:
            logging.error(f"  🔥 抓取時發生嚴重錯誤 {url}: {e}")

    # 4. 整合並「附加」到 CSV
    if new_snapshot_data:
        try:
            new_df = pd.DataFrame(new_snapshot_data, columns=COLUMNS)
            
            # 確保欄位型別正確 (Int64 支援 NaN)
            new_df['headline_level'] = pd.to_numeric(new_df['headline_level'], errors='coerce').astype('Int64')
            new_df['credibility_label'] = pd.to_numeric(new_df['credibility_label'], errors='coerce').astype('Int64')
            
            # *** 關鍵邏輯：附加 (append) 模式 ***
            new_df.to_csv(
                DB_FILENAME, 
                mode='a',                 # 'a' 代表 append (附加)
                header=not file_exists,   # 只有在檔案不存在時才寫入 header
                index=False, 
                encoding='utf-8-sig'
            )
            
            logging.info(f"✅ Part 2 完成：成功抓取並「附加」 {len(new_snapshot_data)} 筆新快照到 {DB_FILENAME}")
            logging.info(f"您的舊資料已完整保留。")
            
        except IOError as e:
            logging.error(f"🔥 Part 2 錯誤：儲存到 {DB_FILENAME} 失敗: {e}")
        except Exception as e:
            logging.error(f"🔥 Part 2 儲存時發生未知錯誤: {e}")
    else:
        logging.warning("👍 Part 2 完成：本次快照未成功抓取到任何新文章。")

logging.info("✅ Cell 6 執行函式已載入 (附加模式)")

2025-10-27 16:44:08,490 - INFO - ✅ Cell 6 執行函式已載入 (附加模式)


In [7]:
# ===============================================================
# Cell 7: 完整執行所有爬蟲任務
# ===============================================================

def main_execution():
    """
    依序執行所有爬蟲步驟
    """
    
    start_time = time.time()
    logging.info(f"======= 爬蟲任務開始於 {get_current_utc_time()} =======")
    
    # 步驟一：抓取所有頭條
    run_part1_headline_fetch()
    
    # 步驟二：抓取新文章內文並附加到您的 news_snapshot_log.csv
    run_part2_scrape_and_append()
    
    # 步驟三：(已移除) 您的日誌檔案是快照模式，不需要 Part 3 的升級邏輯。
    
    end_time = time.time()
    sources = "CNA, UDN, SETN, TVBS, EBC, NOWnews, ChinaTimes, ETtoday, LTN"
    logging.info(f"\n======= 爬蟲任務完成 ({sources})，總耗時: {end_time - start_time:.2f} 秒 =======")

# 確保這個 .py 檔案被直接執行時，才會觸發 main_execution()
if __name__ == "__main__" or "ipykernel" in sys.modules:
    # 立即執行主程式
    main_execution()

2025-10-27 16:44:21,112 - INFO - ======= 爬蟲任務開始於 2025-10-27 16:44:21.112630 =======
2025-10-27 16:44:21,113 - INFO - --- Part 1 開始：抓取所有 9 個網站頭條 ---
2025-10-27 16:44:21,113 - INFO - --- (1/9) 正在抓取 CNA 中央社頭條 ---
2025-10-27 16:44:21,773 - INFO - 💡 CNA 抓取到 0 筆頭條
2025-10-27 16:44:21,773 - INFO - --- (2/9) 正在抓取 UDN 聯合新聞網頭條 ---
2025-10-27 16:44:22,051 - INFO - 💡 UDN 抓取到 0 筆頭條
2025-10-27 16:44:22,051 - INFO - --- (3/9) 正在抓取 SETN 三立新聞網頭條 ---
2025-10-27 16:44:22,172 - INFO - 💡 SETN 抓取到 154 筆頭條
2025-10-27 16:44:22,172 - INFO - --- (4/9) 正在抓取 TVBS 新聞網頭條 ---
2025-10-27 16:44:22,425 - INFO - 💡 TVBS 抓取到 5 筆頭條
2025-10-27 16:44:22,425 - INFO - --- (5/9) 正在抓取 EBC 東森新聞頭條 ---
2025-10-27 16:44:22,663 - INFO - 💡 EBC 抓取到 23 筆頭條
2025-10-27 16:44:22,663 - INFO - --- (6/9) 正在抓取 NOWnews 今日新聞頭條 ---
2025-10-27 16:44:23,004 - INFO - 💡 NOWnews 抓取到 22 筆頭條
2025-10-27 16:44:23,004 - INFO - --- (7/9) 正在抓取 China Times 中時新聞網頭條 ---
2025-10-27 16:44:23,199 - INFO - 💡 China Times 抓取到 7 筆頭條
2025-10-27 16:44:23,200 - INFO - --